In [1]:
import pandas as pd
import pickle

In [ ]:
cbs = pd.read_csv("../data/CBS_dataset_v1.0.csv", sep=';',on_bad_lines="skip")

<positron-console-cell-2>:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.


In [3]:
cbs = pd.read_csv("../data/CBS_dataset_v1.0.csv", sep=',,',on_bad_lines="skip")
general_audience_speeches = pd.DataFrame(pd.read_excel("../data/general_audience_speeches.xlsx"))

<positron-console-cell-3>:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.


In [21]:
cbs = cbs.reset_index(drop=False)
cbs = cbs.rename(columns={"level_2": "Meta", cbs.columns[-1]: "Last"})
meta_split = cbs["Meta"].str.split(",",n=6, expand=True)
meta_split.columns = [
    "Date",
    "Author",
    "Position",
    "Gender",
    "Institution",
    "Country",
    "Speech"
]
cbs = pd.concat(
    [cbs.drop(columns=["Meta"]), meta_split],
    axis=1
)
cbs = cbs.rename(columns={"level_0":"url", "level_1":"title", cbs.columns[3]:"type"})

In [12]:
valid_url = cbs["url"].str.split("/").str[2].drop_duplicates()
valid_url = valid_url.str.replace(" ", "")
valid_url = valid_url[valid_url.str.contains(r"\.", na=False)]
valid_url = valid_url[~valid_url.str.contains(r";", na=False)]
valid_url = valid_url[~valid_url.str.contains(r"^\s*\d+\s*$", na=False)]

cbs["valid_url"] = cbs["url"].str.split("/").str[2].replace(" ", "").dropna()

In [13]:
pattern = r"""(?ix)
\b(
    federalreserve\.gov |
    [a-z]+fed\.org |          # atlantafed.org, chicagofed.org, etc.
    frbsf\.org |

    bankofengland\.co\.uk |

    ecb\.europa\.eu
)\b
"""

filtered = valid_url[valid_url.str.contains(pattern, na=False)]

cbs = cbs[cbs["valid_url"].isin(filtered)]


<positron-console-cell-13>:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.


In [14]:
cols = ["title", "Speech", "Last"]

new_df = (
    cbs[cols]
        .fillna("")                 # remove NaN issues
        .astype(str)                # ensure strings
        .agg(" ".join, axis=1)      # concatenate row-wise
        .str.replace(r"\s+", " ", regex=True)  # clean extra spaces
        .str.strip()
)

In [15]:
import re

W2V_TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")  # keeps words like don't

def tokenize_for_w2v(s: str) -> list[str]:
    s = "" if s is None else str(s)
    s = s.lower()

    # remove urls-ish and email-ish fragments (optional but usually helps)
    s = re.sub(r"\b(?:https?://|www\.)\S+\b", " ", s)
    s = re.sub(r"\b\S+@\S+\b", " ", s)

    # keep alphabetic tokens (drops numbers, punctuation)
    return W2V_TOKEN_RE.findall(s)

# Word2Vec training input: list of token lists (one per row)
sentences_cbs = new_df.fillna("").map(tokenize_for_w2v).tolist()

# optional: drop empty rows
sentences_cbs = [toks for toks in sentences_cbs if toks]

with open("sentences_cbs.pkl", "wb") as f:
    pickle.dump(sentences_cbs, f, protocol=pickle.HIGHEST_PROTOCOL)

In [11]:
general_audience_speeches_tokenized = [tokenize_for_w2v(s) for s in general_audience_speeches['Text']]


with open("general_audience_speeches_tokenized.pkl", "wb") as f:
    pickle.dump(general_audience_speeches_tokenized, f, protocol=pickle.HIGHEST_PROTOCOL)


In [12]:
general_audience_speeches.iloc[83]

Institution                                                  Fed
Date                                         2024-07-31 00:00:00
Text           The Fed has been assigned two goals for moneta...
Name: 83, dtype: object